In [1]:
import pickle
import torch
import json
from tqdm import tqdm
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def scale(x,clip=None):
    if clip:
        return (x).tolist()[:clip]
    else:
        return (x).tolist()
    
def plot(combined,clip=None):
    fig=make_subplots(rows=1,cols=1,subplot_titles=["correct_true-wrong_true cat correct_false-wrong_false"])
    fig.add_trace(go.Scatter(y=scale(combined,clip)),row=1,col=1)
    fig.update_layout(height=400,width=1000,title_text="Singular values, Linear")
    fig.show()
    
def get_simpleqa(layer,post):
    with open(f'/kaggle/input/svd-layers/simpleqa_layer{layer}_{"post" if post==1 else "mid"}.dat','rb') as file:
        data=pickle.load(file)
    return data
def get_halueval(layer):
    with open(f'/kaggle/input/svd-polarity/halueval_layer{layer}_post.dat','rb') as file:
        data=pickle.load(file)
    cap=min(len(data[0][0]),len(data[1][0]))
    data[0][0]=data[0][0][:cap]
    data[0][1]=data[0][1][:cap]
    data[1][0]=data[1][0][:cap]
    data[1][1]=data[1][1][:cap]
    return data
def get_svdvals(data,mean,norm):
    wrong_false  =torch.stack(data[0][0])
    wrong_true   =torch.stack(data[0][1])
    correct_false=torch.stack(data[1][0])
    correct_true =torch.stack(data[1][1])
    if mean==1:
        wrong_false  =wrong_false  -torch.mean(wrong_false  ,dim=1,keepdim=True)
        wrong_true   =wrong_true   -torch.mean(wrong_true   ,dim=1,keepdim=True)
        correct_false=correct_false-torch.mean(correct_false,dim=1,keepdim=True)
        correct_true =correct_true -torch.mean(correct_true ,dim=1,keepdim=True)
    if norm==1:
        wrong_false  =wrong_false  /torch.norm(wrong_false  ,dim=1,keepdim=True)
        wrong_true   =wrong_true   /torch.norm(wrong_true   ,dim=1,keepdim=True)
        correct_false=correct_false/torch.norm(correct_false,dim=1,keepdim=True)
        correct_true =correct_true /torch.norm(correct_true ,dim=1,keepdim=True)
    
    combined=torch.linalg.svdvals(torch.cat((correct_true -wrong_true,correct_false-wrong_false)))
    return combined

In [2]:
layer=22
post =1
mean =0
norm =0
combined=get_svdvals(get_simpleqa(layer,post),mean,norm)
#print((combined[0]/combined[100]).item())
plot(combined)

In [3]:
layer=22
mean =0
norm =0
combined=get_svdvals(get_halueval(layer),mean,norm)
#print((combined[0]/combined[100]).item())
plot(combined)

In [4]:
layer=21
mean =0
norm =0
combined=get_svdvals(get_halueval(layer),mean,norm)
print((combined[0]/combined[100]).item())
plot(combined)

15.175076484680176


In [5]:
ratio_simpleqa=[]
x1=0
x2=100
for i in tqdm(range(26)):
    r=[]
    x=get_svdvals(get_simpleqa(i,1),0,0)
    for j in range(1,x2+1):
        r.append((x[x1]/x[j]).item())
    ratio_simpleqa.append(r)

100%|██████████| 26/26 [00:23<00:00,  1.11it/s]


In [6]:
ratio_halueval=[]
x1=0
x2=100
for i in tqdm(range(26)):
    r=[]
    x=get_svdvals(get_halueval(i),0,0)
    for j in range(1,x2+1):
        r.append((x[x1]/x[j]).item())
    ratio_halueval.append(r)

100%|██████████| 26/26 [01:48<00:00,  4.16s/it]


In [7]:
max_svd_simpleqa=[]
max_svd_halueval=[]
for i in tqdm(range(26)):
    x1=get_svdvals(get_simpleqa(i,0),0,0)
    x2=get_svdvals(get_halueval(i),0,0)
    max_svd_simpleqa.append(x1[0].item())
    max_svd_halueval.append(x2[0].item())

100%|██████████| 26/26 [01:52<00:00,  4.32s/it]


In [8]:
px.imshow(ratio_simpleqa)

In [9]:
px.imshow(ratio_halueval)

In [10]:
px.imshow(torch.abs(torch.tensor(ratio_simpleqa)-torch.tensor(ratio_halueval)))

In [11]:
px.line(list(zip(*ratio_halueval)))

In [12]:
px.line(list(zip(*ratio_simpleqa)))

In [13]:
px.line(max_svd_simpleqa)

In [14]:
px.line(max_svd_halueval)